# Example 7: Training a Virtual Room for Feedback Cancellation

This notebook demonstrates training a **phase‑cancelling modal reverberator** to suppress feedback in a Reverberation Enhancement System (RES). The virtual room is a modal reverb whose modes can be tuned to cancel the feedback paths measured in a real room (loaded from the DataRES dataset).

**Training pipeline** (based on De Bortoli et al., JAES 2024):
1. Load a physical room from DataRES (real measurements).
2. Create a virtual room as a phase‑cancelling modal reverb (trainable mode gains and phases).
3. Build the RES and extract the open‑loop model.
4. Dataset: unit impulses as inputs, zeros as targets (we want to minimise the open‑loop eigenvalues at the mode frequencies).
5. Two loss functions:  
   - `MSE_evs_idxs` minimises the eigenvalue magnitude at the modal frequencies.  
   - `colorless_reverb` encourages the reverb’s energy to remain as high as at initialisation (to avoid simply turning it off).
6. Train and compare eigenvalues/impulse responses before and after.

The example uses a downsampled rate (1000 Hz) for faster experimentation. You can adjust all hyperparameters below.

## 1. Imports and Setup

In [ ]:
import sys
import os
import time
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..')))

import torch
import matplotlib.pyplot as plt

from flamo import system, dsp
from flamo.optimize.dataset import Dataset, load_dataset
from flamo.optimize.trainer import Trainer

from PyRES.res import RES
from PyRES.physical_room import PhRoom_dataset
from PyRES.virtual_room import phase_cancellation
from PyRES.loss_functions import MSE_evs_idxs, colorless_reverb
from PyRES.plots import plot_evs_compare, plot_spectrograms_compare

torch.manual_seed(141122)

## 2. Hyperparameters

In [ ]:
# ----------------------- Dataset -----------------------
num = 2**4                 # dataset size
device = 'cpu'             # computation device
split = 0.8                # train/validation split

# ---------------------- Training -----------------------
max_epochs = 10
patience_delta = 1e-4
lr = 1e-3

# ---------------------- Output -------------------------
train_dir = None           # if None, a timestamped folder is created

# ---------------------- Room data -----------------------
dataset_directory = './dataRES'   # path to your DataRES folder
room_name = 'Otala'                # room to load

# ---------------------- Modal reverb --------------------
MR_n_modes = 120           # number of modes
MR_f_low = 50              # lowest mode frequency (Hz)
MR_f_high = 450            # highest mode frequency (Hz)
MR_t60 = 1.0               # reverberation time (s)

## 3. Load Physical Room (DataRES)

In [ ]:
# Time‑frequency parameters (downsampled for speed)
samplerate = 1000           # Hz (note: lower than before)
nfft = samplerate * 3       # FFT size
alias_decay_db = -20        # anti‑aliasing decay (dB)

physical_room = PhRoom_dataset(
    fs=samplerate,
    nfft=nfft,
    alias_decay_db=alias_decay_db,
    dataset_directory=dataset_directory,
    room_name=room_name
)

n_M = physical_room.transducer_number['mcs']   # number of microphones
n_L = physical_room.transducer_number['lds']   # number of loudspeakers

print(f"Loaded room '{room_name}' with {n_M} microphones and {n_L} loudspeakers.")

## 4. Create Virtual Room (Phase‑Cancelling Modal Reverb)

In [ ]:
virtual_room = phase_cancellation(
    n_M=n_M,
    n_L=n_L,
    fs=samplerate,
    nfft=nfft,
    n_modes=MR_n_modes,
    low_f_lim=MR_f_low,
    high_f_lim=MR_f_high,
    t60=MR_t60,
    requires_grad=True,
    alias_decay_db=alias_decay_db
)

# Build the RES
res = RES(physical_room=physical_room, virtual_room=virtual_room)

## 5. Define the Model (Open Loop)

In [ ]:
model = system.Shell(
    core=res.open_loop(),
    input_layer=system.Series(
        dsp.FFT(nfft=nfft),
        dsp.Transform(lambda x: x.diag_embed())
    )
)

## 6. Initial Performance

In [ ]:
evs_init = res.open_loop_eigenvalues()
_, _, ir_init = res.system_simulation()   # ir_init shape: (samples, receivers) – here receivers = audience

## 7. Create Dataset

Input: unit impulses in the frequency domain (first bin = 1).  
Target: zeros – we want to suppress the open‑loop eigenvalues.

In [ ]:
# Input: frequency‑domain impulses (first bin = 1, others 0)
dataset_input = torch.zeros(1, nfft // 2 + 1, n_M)
dataset_input[:, 0, :] = 1.0

dataset_target = torch.zeros(1, nfft // 2 + 1, n_M)

dataset = Dataset(
    input=dataset_input,
    target=dataset_target,
    expand=num,
    device=device
)

train_loader, valid_loader = load_dataset(dataset, batch_size=1, split=split, shuffle=False)

## 8. Set Up Trainer and Loss Functions

In [ ]:
if train_dir is None:
    train_dir = os.path.join('training_output', time.strftime("%Y%m%d-%H%M%S"))
os.makedirs(train_dir, exist_ok=True)

# Save hyperparameters
with open(os.path.join(train_dir, 'args.txt'), 'w') as f:
    f.write('\n'.join([f"{k},{v}" for k, v in locals().items() if k in ['num','device','split','max_epochs','patience_delta','lr','MR_n_modes','MR_f_low','MR_f_high','MR_t60']]))

trainer = Trainer(
    net=model,
    max_epochs=max_epochs,
    lr=lr,
    patience_delta=patience_delta,
    train_dir=train_dir,
    device=device
)

# Loss 1: minimise eigenvalues at the mode frequencies
MR_freqs = virtual_room.get_v_ML()[0].resonances[:, 0, 0].clone().detach()   # shape (n_modes,)
criterion1 = MSE_evs_idxs(
    iter_num=num,
    freq_points=nfft // 2 + 1,
    samplerate=samplerate,
    freqs=MR_freqs
)
trainer.register_criterion(criterion1, 1.0)

# Loss 2: keep the reverb’s energy close to its initial value (to avoid trivial zero solution)
criterion2 = colorless_reverb(
    samplerate=samplerate,
    freq_points=nfft // 2 + 1,
    freqs=MR_freqs
)
trainer.register_criterion(criterion2, 0.2, requires_model=True)

## 9. Run Training

In [ ]:
trainer.train(train_loader, valid_loader)

## 10. Evaluate After Training

In [ ]:
evs_opt = res.open_loop_eigenvalues()
_, _, ir_opt = res.system_simulation()

# Compare eigenvalues (zoomed to mode frequency range)
plot_evs_compare(evs_init, evs_opt, samplerate, nfft, 40, 460)
plt.show()

# Compare impulse responses (first audience channel)
plot_spectrograms_compare(
    ir_init,            # shape (samples, receivers) – we take all channels together
    ir_opt,
    fs=samplerate,
    nfft=2**4,
    noverlap=2**3
)
plt.show()

## 11. (Optional) Save Trained Model

In [ ]:
# res.save_state_to(directory='./model_states/')

## 12. Conclusion

The trained modal reverb now exhibits reduced open‑loop eigenvalues at its modal frequencies, indicating feedback cancellation. The second loss ensures that the reverberator still contributes energy. This proof‑of‑concept can be scaled by increasing the dataset size and training duration.

Reference:
> De Bortoli, G., Prawda, K., and Schlecht, S. J.  
> "Active Acoustics with a Phase Cancelling Modal Reverberator"  
> *Journal of the Audio Engineering Society*, Vol. 72, No. 10, pp. 705‑715, 2024.